# WiDS Datathon 2026 - Wildfire Prediction

## Notebook 2: Data Visualization

---

### 📊 Purpose of This Notebook

This notebook creates **comprehensive visualizations** to understand the wildfire prediction dataset. Visualization is crucial in data science because:

1. **Reveals Patterns**: Visual patterns are often easier to spot than numerical patterns
2. **Validates Assumptions**: Confirms or challenges our understanding of the data
3. **Communicates Insights**: Makes findings accessible to non-technical stakeholders
4. **Guides Modeling**: Helps identify which features and approaches will work best

### 🎯 What We'll Visualize

This notebook creates comprehensive visualizations to understand:

- **Target variable distributions and survival curves**: How fires reach evacuation zones over time
- **Feature distributions and relationships**: Understanding individual feature behavior
- **Correlation patterns**: Which features move together and predict outcomes
- **Temporal patterns**: Time-based trends (hour, day, month)
- **Category-wise feature analysis**: Grouped analysis by feature type (distance, growth, etc.)

### 💡 Key Visualization Concepts

**Survival Curves (Kaplan-Meier)**:
- Shows probability that fire stays AWAY from evacuation zone over time
- Handles censored data (fires that never reached zones)
- Essential for understanding time-to-event patterns

**Cumulative Incidence**:
- Complement of survival (probability fire REACHES zone by time t)
- Directly relates to our prediction task
- Shows risk accumulation over time

**Multi-Horizon Analysis**:
- Visualizes predictions at 12h, 24h, 48h, 72h
- Must satisfy monotonicity: prob_12h ≤ prob_24h ≤ prob_48h ≤ prob_72h
- Critical for competition submission

## 1. Setup and Data Loading

### 🔧 Why This Setup Matters

**Visualization Configuration**:
- We set specific styles and colors to make plots professional and consistent
- Default figure sizes ensure plots are readable without being too large
- Seaborn's 'husl' palette provides colorblind-friendly, distinct colors

**Reproducibility**:
- Random seed ensures any random elements in visualizations are consistent
- Important for creating reproducible reports and presentations

In [ ]:
# Import libraries
import pandas as pd  # Data manipulation and analysis
import numpy as np  # Numerical operations
import matplotlib.pyplot as plt  # Core plotting library
import seaborn as sns  # Statistical visualization (built on matplotlib)
from scipy import stats  # Statistical functions (for advanced analysis)
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

# Set random seed for reproducibility
# This ensures any random elements (like sampling) produce the same results
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Configure visualization settings
# These settings create professional, consistent plots throughout the notebook
plt.style.use('seaborn-v0_8-darkgrid')  # Clean style with subtle grid
sns.set_palette('husl')  # Colorblind-friendly palette with distinct colors
plt.rcParams['figure.figsize'] = (12, 6)  # Default figure size (width, height in inches)
plt.rcParams['font.size'] = 10  # Base font size for all text in plots

# Display settings for pandas DataFrames
pd.set_option('display.max_columns', None)  # Show all columns (no truncation)
pd.set_option('display.max_rows', 100)  # Show up to 100 rows

print("Libraries imported successfully")

In [ ]:
# Load data
# We load the same datasets as in the EDA notebook for consistency
train_df = pd.read_csv('../data/train.csv')  # Training data with targets
test_df = pd.read_csv('../data/test.csv')  # Test data (no targets)
metadata_df = pd.read_csv('../data/metaData.csv')  # Feature descriptions

# Display dataset dimensions
# Shape is (rows, columns) - helps verify data loaded correctly
print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"Metadata shape: {metadata_df.shape}")

In [ ]:
# Define feature categories based on metadata
# Organizing features by domain helps us create focused, interpretable visualizations
# Each category represents a different aspect of wildfire behavior
feature_categories = {
    # Temporal coverage: How many observations and their time span
    'temporal_coverage': ['num_perimeters_0_5h', 'dt_first_last_0_5h', 'low_temporal_resolution_0_5h'],
    
    # Growth: How the fire is expanding (area, rate, etc.)
    'growth': ['area_first_ha', 'area_growth_abs_0_5h', 'area_growth_rel_0_5h', 'area_growth_rate_ha_per_h',
               'log1p_area_first', 'log1p_growth', 'log_area_ratio_0_5h', 'relative_growth_0_5h',
               'radial_growth_m', 'radial_growth_rate_m_per_h'],
    
    # Centroid kinematics: Movement of fire's center point
    'centroid_kinematics': ['centroid_displacement_m', 'centroid_speed_m_per_h', 'spread_bearing_deg',
                            'spread_bearing_sin', 'spread_bearing_cos'],
    
    # Distance: How far fire is from evacuation zone and how that's changing
    'distance': ['dist_min_ci_0_5h', 'dist_std_ci_0_5h', 'dist_change_ci_0_5h', 'dist_slope_ci_0_5h',
                 'closing_speed_m_per_h', 'closing_speed_abs_m_per_h', 'projected_advance_m',
                 'dist_accel_m_per_h2', 'dist_fit_r2_0_5h'],
    
    # Directionality: Is fire moving toward or away from evacuation zone?
    'directionality': ['alignment_cos', 'alignment_abs', 'cross_track_component', 'along_track_speed'],
    
    # Temporal metadata: When did the fire start? (hour, day, month)
    'temporal_metadata': ['event_start_hour', 'event_start_dayofweek', 'event_start_month']
}

# Separate features and targets
# Target columns are what we're trying to predict
target_cols = ['time_to_hit_hours', 'event']  # Time until hit + whether it hit
# Feature columns are everything except ID and targets
feature_cols = [col for col in train_df.columns if col not in ['event_id'] + target_cols]

print(f"Total features: {len(feature_cols)}")
print(f"Target columns: {target_cols}")

## 2. Target Variable Visualization

### 🎯 Understanding Our Targets

We have **two target variables** that work together in survival analysis:

1. **`event`** (binary): Did the fire reach within 5km? (1=Yes, 0=No/Censored)
2. **`time_to_hit_hours`** (continuous): When did it reach 5km? (0-72 hours)

### 📊 Why These Visualizations Matter

**Event Distribution**:
- Shows the balance between events (fires that hit) and censored observations (didn't hit)
- ~50% censoring is common in survival analysis
- Helps us understand the challenge: we need to predict both IF and WHEN

**Time Distribution**:
- Shows WHEN fires typically reach evacuation zones
- Helps identify critical time windows (e.g., most hits in first 24h?)
- Guides our multi-horizon predictions (12h, 24h, 48h, 72h)

**Box Plot Comparison**:
- Compares time distributions between hit and censored events
- Censored events have time=72h (end of observation window)
- Hit events show actual time when fire reached zone

### 💡 What to Look For

- **Event balance**: Are hits and censored roughly equal?
- **Time patterns**: Do most hits occur early or late in the 72h window?
- **Median time**: What's the typical time for fires that do hit?
- **Distribution shape**: Uniform? Skewed? Multiple peaks?

In [ ]:
# Target variable distributions
# Create a 2x2 grid of subplots for comprehensive target analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# ============================================================================
# PLOT 1: Event Distribution (Top Left)
# ============================================================================
# Shows the balance between fires that hit (event=1) and didn't hit (event=0)
event_counts = train_df['event'].value_counts()
axes[0, 0].bar(['Censored (0)', 'Hit (1)'], event_counts.values, color=['#FF6B6B', '#4ECDC4'])
axes[0, 0].set_title('Event Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Count')
# Add count and percentage labels on top of bars for easy reading
for i, v in enumerate(event_counts.values):
    axes[0, 0].text(i, v + 5, f'{v} ({v/len(train_df)*100:.1f}%)', ha='center', fontweight='bold')

# ============================================================================
# PLOT 2: Time Distribution - All Events (Top Right)
# ============================================================================
# Shows time distribution for ALL observations (both hit and censored)
# Note: Censored events have time=72h (end of observation window)
axes[0, 1].hist(train_df['time_to_hit_hours'], bins=30, color='#95E1D3', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Time to Hit Distribution (All Events)', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Time to Hit (hours)')
axes[0, 1].set_ylabel('Frequency')
# Add median line for reference (helps identify central tendency)
axes[0, 1].axvline(train_df['time_to_hit_hours'].median(), color='red', linestyle='--', 
                    label=f'Median: {train_df["time_to_hit_hours"].median():.1f}h')
axes[0, 1].legend()

# ============================================================================
# PLOT 3: Time Distribution - Hit Events Only (Bottom Left)
# ============================================================================
# Shows ACTUAL time-to-hit for fires that reached evacuation zones
# This excludes censored observations to see true event timing
hit_events = train_df[train_df['event'] == 1]['time_to_hit_hours']
axes[1, 0].hist(hit_events, bins=20, color='#F38181', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Time to Hit Distribution (Hit Events Only)', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Time to Hit (hours)')
axes[1, 0].set_ylabel('Frequency')
# Add median line for hit events
axes[1, 0].axvline(hit_events.median(), color='darkred', linestyle='--', 
                    label=f'Median: {hit_events.median():.1f}h')
axes[1, 0].legend()

# ============================================================================
# PLOT 4: Box Plot Comparison (Bottom Right)
# ============================================================================
# Compares time distributions between censored (0) and hit (1) events
# Box plot shows: median (line), quartiles (box), and outliers (points)
train_df.boxplot(column='time_to_hit_hours', by='event', ax=axes[1, 1])
axes[1, 1].set_title('Time to Hit by Event Status', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Event (0=Censored, 1=Hit)')
axes[1, 1].set_ylabel('Time to Hit (hours)')
plt.suptitle('')  # Remove default title from boxplot

# Save and display the figure
plt.tight_layout()  # Adjust spacing to prevent overlap
plt.savefig('../figures/target_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

# Print summary statistics for interpretation
print(f"\nTarget Variable Statistics:")
print(f"Event=1 (Hit): {event_counts[1]} ({event_counts[1]/len(train_df)*100:.1f}%)")
print(f"Event=0 (Censored): {event_counts[0]} ({event_counts[0]/len(train_df)*100:.1f}%)")
print(f"\nTime to Hit (All): Mean={train_df['time_to_hit_hours'].mean():.2f}h, Median={train_df['time_to_hit_hours'].median():.2f}h")
print(f"Time to Hit (Hit Events): Mean={hit_events.mean():.2f}h, Median={hit_events.median():.2f}h")
print(f"\n💡 Key Observation: Compare the median times to understand typical fire behavior")

## 3. Kaplan-Meier Survival Curve

### 📈 What is a Kaplan-Meier Curve?

The **Kaplan-Meier (KM) estimator** is the gold standard for survival analysis. It shows the probability that an event has NOT occurred by time t.

**In our context**:
- **Survival** = Fire stays BEYOND 5km from evacuation zone
- **Event** = Fire comes WITHIN 5km (what we're trying to predict)
- **Censored** = Fire never reached 5km during 72h observation window

### 🔑 Key Concepts

**Why KM Curve Matters**:
1. **Handles Censored Data**: Unlike simple averages, KM properly accounts for fires that never hit
2. **Shows Risk Over Time**: Reveals when fires are most likely to reach zones
3. **Guides Predictions**: Helps calibrate our multi-horizon probability forecasts

**How to Read the Curve**:
- **Y-axis**: Probability fire stays away (1.0 = 100% safe, 0.0 = 0% safe)
- **X-axis**: Time in hours (0 to 72)
- **Downward steps**: Each step down = fires reaching evacuation zones
- **Flat sections**: No events occurring during that time period
- **Median survival time**: When curve crosses 50% line (half of fires have hit)

### 💡 What to Look For

- **Initial drop**: Do many fires hit quickly (steep early decline)?
- **Curve shape**: Steady decline or sudden drops?
- **Final probability**: What % of fires never reach zones?
- **Median time**: Typical time for fires that do hit

In [ ]:
# Calculate Kaplan-Meier survival curve
def calculate_kaplan_meier(time, event):
    """
    Calculate Kaplan-Meier survival curve.
    
    Parameters:
    - time: array of event times
    - event: array of event indicators (1=event occurred, 0=censored)
    
    Returns:
    - time_points: unique time points
    - survival_prob: survival probability at each time point
    """
    # Combine time and event into a dataframe
    df = pd.DataFrame({'time': time, 'event': event})
    df = df.sort_values('time')
    
    # Get unique time points
    time_points = df['time'].unique()
    
    # Calculate survival probability
    n_at_risk = len(df)
    survival_prob = [1.0]
    times = [0]
    
    for t in time_points:
        # Number of events at time t
        n_events = df[(df['time'] == t) & (df['event'] == 1)].shape[0]
        # Number at risk (all with time >= t)
        n_at_risk = df[df['time'] >= t].shape[0]
        
        if n_at_risk > 0:
            # Update survival probability
            survival_prob.append(survival_prob[-1] * (1 - n_events / n_at_risk))
            times.append(t)
    
    return np.array(times), np.array(survival_prob)

# Calculate KM curve using our function
# This gives us survival probabilities at each time point
km_times, km_survival = calculate_kaplan_meier(
    train_df['time_to_hit_hours'].values,
    train_df['event'].values
)

# ============================================================================
# Plot Kaplan-Meier Survival Curve
# ============================================================================
fig, ax = plt.subplots(figsize=(12, 6))

# Plot the survival curve as a step function
# 'where=post' creates steps that drop at event times (standard for KM curves)
ax.step(km_times, km_survival, where='post', linewidth=2.5, color='#4ECDC4', label='Kaplan-Meier Estimate')
# Fill area under curve for better visualization
ax.fill_between(km_times, km_survival, step='post', alpha=0.3, color='#4ECDC4')

# Labels and title
ax.set_xlabel('Time (hours)', fontsize=12, fontweight='bold')
ax.set_ylabel('Survival Probability (Fire NOT within 5km)', fontsize=12, fontweight='bold')
ax.set_title('Kaplan-Meier Survival Curve\n(Probability Fire Stays Beyond 5km)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
ax.set_xlim(0, 72)  # Full observation window
ax.set_ylim(0, 1.05)  # Slightly above 1.0 for better visibility

# Add reference lines for median survival time
# Median = time when 50% of fires have hit evacuation zones
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='50% Survival')
median_time_idx = np.where(km_survival <= 0.5)[0]
if len(median_time_idx) > 0:
    median_time = km_times[median_time_idx[0]]  # type: ignore[index]
    ax.axvline(x=median_time, color='red', linestyle='--', alpha=0.5)  # type: ignore[arg-type]
    ax.text(median_time + 2, 0.55, f'Median: {median_time:.1f}h', fontsize=10, color='red')  # type: ignore[arg-type]

plt.tight_layout()
plt.savefig('../figures/kaplan_meier_curve.png', dpi=300, bbox_inches='tight')
plt.show()

# Print key statistics
print(f"\nSurvival Analysis:")
print(f"Initial survival probability: {km_survival[0]:.3f} (100% at start)")
print(f"Final survival probability (at 72h): {km_survival[-1]:.3f}")
print(f"  → {(1-km_survival[-1])*100:.1f}% of fires reached evacuation zones")
if len(median_time_idx) > 0:
    print(f"Median survival time: {median_time:.2f} hours")  # type: ignore[possibly-unbound]
    print(f"  → Half of fires that hit did so by {median_time:.1f}h")  # type: ignore[possibly-unbound]

## 3b. Cumulative Incidence Curve (Multi-Horizon Probabilities)

### 📊 From Survival to Risk

**Cumulative Incidence** is the complement of survival:
- **Survival** = P(fire stays away)
- **Cumulative Incidence** = P(fire reaches zone by time t) = 1 - Survival

**Why This Matters for Our Competition**:
- We need to predict **probabilities of hitting** at 12h, 24h, 48h, 72h
- Cumulative incidence directly gives us these probabilities
- Must satisfy monotonicity: prob_12h ≤ prob_24h ≤ prob_48h ≤ prob_72h

### 🎯 Multi-Horizon Predictions

The competition requires predictions at **four time horizons**:
1. **12h**: Short-term (immediate threat assessment)
2. **24h**: Medium-term (operational planning)
3. **48h**: Extended forecast (resource allocation)
4. **72h**: Long-term (strategic decisions)

**Evaluation Weights** (in Brier Score component):
- 12h: 20%
- 24h: 20%
- 48h: 40% ← Highest weight!
- 72h: 20%

In [ ]:
# Calculate cumulative incidence (complement of survival)
# This transforms "probability of staying away" into "probability of hitting"
cumulative_incidence = 1 - km_survival

# Print survival probabilities at key time horizons
# These are the exact time points we need to predict for submission
print("\nSurvival Probabilities at Key Time Horizons:")
print("(These are the probabilities we need to predict for competition submission)\n")
horizons = [12, 24, 48, 72]
for horizon in horizons:
    # Find the survival probability at or just before this horizon
    idx = np.where(km_times <= horizon)[0]
    if len(idx) > 0:
        surv_prob = km_survival[idx[-1]]  # Probability fire stays away
        hit_prob = 1 - surv_prob  # Probability fire reaches zone (what we predict)
        print(f"  {horizon}h: Survival={surv_prob:.3f}, Hit Probability={hit_prob:.3f}")
        print(f"       → {hit_prob*100:.1f}% chance fire reaches evacuation zone by {horizon}h")

In [ ]:
# ============================================================================
# Plot Cumulative Incidence with Submission Horizons
# ============================================================================
# This plot shows the probability of fire REACHING evacuation zone by time t
fig, ax = plt.subplots(figsize=(14, 7))

# Plot cumulative incidence curve
# This is the inverse of survival - shows increasing risk over time
ax.step(km_times, cumulative_incidence, where='post', linewidth=2.5, color='#F38181', 
        label='Cumulative Incidence', zorder=2)
ax.fill_between(km_times, cumulative_incidence, step='post', alpha=0.3, color='#F38181', zorder=1)

# Add markers for key time horizons (submission requirements)
# These are the exact probabilities we need to predict for competition
horizon_colors = ['#FFD93D', '#6BCB77', '#4D96FF', '#9D4EDD']  # Distinct colors for each horizon
horizon_probs = []  # Store probabilities for later analysis

for horizon, color in zip(horizons, horizon_colors):
    # Find cumulative incidence at this horizon
    idx = np.where(km_times <= horizon)[0]
    if len(idx) > 0:
        prob = cumulative_incidence[idx[-1]]  # type: ignore[index] # Probability at this time point
        horizon_probs.append(prob)
        
        # Add large marker at this point
        ax.plot(horizon, prob, 'o', markersize=14, color=color,  # type: ignore[arg-type]
                markeredgecolor='black', markeredgewidth=2.5, 
                label=f'{horizon}h: {prob:.3f}', zorder=3)
        
        # Add reference lines to make values easy to read
        ax.axvline(x=horizon, color=color, linestyle=':', alpha=0.6, linewidth=2, zorder=1)
        ax.axhline(y=prob, color=color, linestyle=':', alpha=0.3, linewidth=1.5, zorder=1)  # type: ignore[arg-type]

# Labels and formatting
ax.set_xlabel('Time (hours)', fontsize=13, fontweight='bold')
ax.set_ylabel('Cumulative Incidence\n(Probability Fire Within 5km)', fontsize=13, fontweight='bold')
ax.set_title('Cumulative Incidence Curve with Submission Horizons\n(Probability Fire Reaches Within 5km by Time t)', 
             fontsize=15, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, zorder=0)
ax.legend(fontsize=11, loc='lower right', framealpha=0.95)
ax.set_xlim(0, 72)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('../figures/cumulative_incidence_with_horizons.png', dpi=300, bbox_inches='tight')
plt.show()

# Print the exact probabilities we need to predict
print("\nCumulative Incidence at Submission Horizons:")
print("(These are the target probabilities for our predictions)\n")
for horizon, prob in zip(horizons, horizon_probs):
    print(f"  prob_{horizon}h: {prob:.4f} ({prob*100:.2f}%)")

## 3c. Multi-Horizon Probability Analysis

### 📊 Three Views of the Same Data

This section provides three complementary visualizations:

1. **Evolution Over Time**: Shows how probability accumulates across the full 72h window
2. **Horizon Comparison**: Direct comparison of probabilities at each submission point
3. **Incremental Analysis**: Shows probability INCREASE between consecutive horizons

### 💡 Why Multiple Views?

- **Evolution plot**: Helps understand the overall risk trajectory
- **Bar chart**: Makes it easy to compare magnitudes across horizons
- **Incremental plot**: Reveals which time periods have highest risk increase

### ✅ Monotonicity Check

**Critical Requirement**: Probabilities MUST increase (or stay same) over time
- If prob_24h < prob_12h → Invalid submission!
- This is a logical constraint: risk can't decrease as time passes
- Our KM-based probabilities naturally satisfy this (by construction)

In [ ]:
# ============================================================================
# Three-Panel Multi-Horizon Analysis
# ============================================================================
# Visualize how probabilities evolve across the four submission horizons
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ============================================================================
# PANEL 1: Probability Evolution Over Time (Left)
# ============================================================================
# Shows the full cumulative incidence curve with horizon markers
ax = axes[0]
ax.step(km_times, cumulative_incidence, where='post', linewidth=2.5, color='#4ECDC4', zorder=2)
ax.fill_between(km_times, cumulative_incidence, step='post', alpha=0.3, color='#4ECDC4', zorder=1)

# Add markers and labels for each submission horizon
for horizon, color in zip(horizons, horizon_colors):
    idx = np.where(km_times <= horizon)[0]
    if len(idx) > 0:
        prob = cumulative_incidence[idx[-1]]
        # Vertical line at horizon time
        ax.axvline(x=horizon, color=color, linestyle='--', alpha=0.6, linewidth=2, zorder=1)
        # Marker at probability value
        ax.plot(horizon, prob, 'o', markersize=10, color=color, 
                markeredgecolor='black', markeredgewidth=2, zorder=3)
        # Text label with time and probability
        ax.text(horizon, prob + 0.05, f'{horizon}h\n{prob:.3f}', 
                ha='center', fontsize=9, fontweight='bold', 
                bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.7))

ax.set_xlabel('Time (hours)', fontsize=11, fontweight='bold')
ax.set_ylabel('Cumulative Hit Probability', fontsize=11, fontweight='bold')
ax.set_title('Probability Evolution Over Time', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, zorder=0)
ax.set_xlim(0, 72)
ax.set_ylim(0, 1.05)

# ============================================================================
# PANEL 2: Horizon Comparison Bar Chart (Middle)
# ============================================================================
# Direct comparison of cumulative probabilities at each horizon
ax = axes[1]
bars = ax.bar(range(len(horizons)), horizon_probs, 
              color=horizon_colors,
              edgecolor='black', linewidth=2, alpha=0.85)
ax.set_xticks(range(len(horizons)))
ax.set_xticklabels([f'{h}h' for h in horizons], fontsize=11, fontweight='bold')
ax.set_xlabel('Submission Time Horizon', fontsize=11, fontweight='bold')
ax.set_ylabel('Cumulative Hit Probability', fontsize=11, fontweight='bold')
ax.set_title('Probabilities at Each Horizon', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y', zorder=0)
ax.set_ylim(0, 1.0)

# Add value labels on top of bars (probability and percentage)
for i, (bar, prob) in enumerate(zip(bars, horizon_probs)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{prob:.4f}\n({prob*100:.1f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# ============================================================================
# PANEL 3: Incremental Probability Increases (Right)
# ============================================================================
# Shows how much probability INCREASES between consecutive horizons
# Helps identify which time periods have highest risk increase
ax = axes[2]
# Calculate increments: first is 0→12h, then 12→24h, 24→48h, 48→72h
increments = [horizon_probs[0]] + [horizon_probs[i] - horizon_probs[i-1] for i in range(1, len(horizon_probs))]
bars = ax.bar(range(len(horizons)), increments, 
              color=horizon_colors,
              edgecolor='black', linewidth=2, alpha=0.85)
ax.set_xticks(range(len(horizons)))
# Label with time intervals
ax.set_xticklabels([f'0→{horizons[0]}h'] + [f'{horizons[i-1]}→{horizons[i]}h' for i in range(1, len(horizons))], 
                    fontsize=9, fontweight='bold', rotation=15, ha='right')
ax.set_xlabel('Time Interval', fontsize=11, fontweight='bold')
ax.set_ylabel('Incremental Probability', fontsize=11, fontweight='bold')
ax.set_title('Probability Increase per Interval', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y', zorder=0)

# Add value labels showing increment size
for i, (bar, inc) in enumerate(zip(bars, increments)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
            f'{inc:.4f}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/multi_horizon_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================================
# Monotonicity Validation
# ============================================================================
# Check that probabilities increase (or stay same) over time
# This is a REQUIRED constraint for valid competition submissions
print("\nMonotonicity Check (probabilities must increase for valid submission):")
print("(Competition will reject submissions that violate this constraint)\n")
monotonic = True
for i in range(len(horizons)-1):
    increase = horizon_probs[i+1] - horizon_probs[i]
    status = "✓ PASS" if increase >= 0 else "✗ FAIL"
    print(f"  {horizons[i]}h → {horizons[i+1]}h: +{increase:.4f} {status}")
    if increase < 0:
        monotonic = False

print(f"\nOverall Monotonicity: {'✓ VALID' if monotonic else '✗ INVALID'}")
if monotonic:
    print("  → Probabilities satisfy monotonicity constraint")
else:
    print("  → WARNING: Probabilities violate monotonicity! Submission would be rejected!")

### Key Insights from Multi-Horizon Analysis:

1. **Probability Distribution**:
   - The cumulative incidence shows how the probability of fire reaching evacuation zones increases over time
   - The four submission horizons (12h, 24h, 48h, 72h) represent critical decision points

2. **Monotonicity Requirement**:
   - Competition requires: `prob_12h ≤ prob_24h ≤ prob_48h ≤ prob_72h`
   - This constraint ensures predictions are logically consistent over time

3. **Operational Context**:
   - 48h horizon has highest weight (40%) in the Brier Score component
   - This reflects the operational value of 24-48 hour forecasts for emergency planning

4. **Modeling Implications**:
   - Models must capture non-linear time dynamics
   - Calibration is critical (70% of score depends on Brier Score)
   - Consider separate models or post-processing for each horizon

## 4. Feature Distribution Visualizations

### 📊 Understanding Feature Distributions

**Why visualize distributions?**
- **Identify skewness**: Right-skewed distributions may need log transforms
- **Spot outliers**: Extreme values that may need special handling
- **Understand scale**: Features with very different ranges need scaling
- **Detect patterns**: Bimodal or unusual distributions reveal data structure

**Key features to examine**:
- **Distance metrics**: How far are fires from evacuation zones?
- **Growth metrics**: How fast are fires expanding?
- **Speed metrics**: How fast are fires moving?
- **Alignment**: Are fires moving toward zones?

**Note**: We exclude zeros for clearer visualization since many features are zero-inflated.

In [ ]:
# Visualize distributions of key numerical features
# These features were identified as most important in EDA
key_features = [
    'area_first_ha',                # Initial fire size
    'area_growth_rate_ha_per_h',    # How fast fire is growing
    'dist_min_ci_0_5h',             # Distance to evacuation zone
    'closing_speed_m_per_h',        # Speed toward zone
    'centroid_speed_m_per_h',       # Overall fire movement speed
    'alignment_abs'                 # Direction alignment with zone
]

fig, axes = plt.subplots(3, 2, figsize=(15, 12))
axes = axes.flatten()

for idx, feature in enumerate(key_features):
    # Remove zeros for better visualization
    # Many features are zero-inflated, so we focus on non-zero values
    data = train_df[train_df[feature] > 0][feature]
    
    # Create histogram
    axes[idx].hist(data, bins=30, color='#95E1D3', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{feature}\n(excluding zeros)', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Value')
    axes[idx].set_ylabel('Frequency')
    
    # Add statistics: mean and median
    # Mean (red): Average value, pulled by outliers
    # Median (blue): Middle value, robust to outliers
    mean_val = data.mean()
    median_val = data.median()
    axes[idx].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
    axes[idx].axvline(median_val, color='blue', linestyle='--', linewidth=2, label=f'Median: {median_val:.2f}')
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/key_feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💡 Observations to look for:")
print("   - Right-skewed distributions (long tail to right) → may need log transform")
print("   - Mean >> Median → presence of outliers pulling mean up")
print("   - Bimodal distributions → two distinct fire behaviors")

## 5. Correlation Analysis

### 🔗 Understanding Feature Relationships

**Why analyze correlations?**
- **Identify predictive features**: Features strongly correlated with target are good predictors
- **Detect multicollinearity**: Highly correlated features provide redundant information
- **Guide feature engineering**: Correlated features can be combined into interactions
- **Understand data structure**: Reveals underlying patterns and relationships

**Correlation coefficient (r)**:
- **r = 1**: Perfect positive correlation (both increase together)
- **r = 0**: No linear relationship
- **r = -1**: Perfect negative correlation (one increases, other decreases)
- **|r| > 0.7**: Strong correlation
- **|r| > 0.9**: Very strong (potential multicollinearity)

**Note**: We focus on features with <50% zeros for more reliable correlation estimates.

In [ ]:
# Correlation heatmap with targets
# Select features with low zero percentage for better correlation analysis
# Zero-inflated features can have misleading correlations
zero_pct = (train_df[feature_cols] == 0).sum() / len(train_df) * 100
features_for_corr = zero_pct[zero_pct < 50].index.tolist()

print(f"Analyzing correlations for {len(features_for_corr)} features with <50% zeros")

# Add targets to correlation analysis
corr_cols = features_for_corr + target_cols
corr_matrix = train_df[corr_cols].corr(method='pearson')  # type: ignore[call-arg]

# Plot correlation heatmap
# Upper triangle masked to avoid redundancy (correlation is symmetric)
fig, ax = plt.subplots(figsize=(16, 14))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix (Features with <50% Zeros)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# Top correlations with targets
print("\n📊 Top 10 Correlations with 'event' (whether fire hit zone):")
event_corr = corr_matrix['event'].drop('event').abs().sort_values(ascending=False).head(10)  # type: ignore[call-overload]
print(event_corr)
print("\n💡 Higher correlation = better predictor of whether fire will hit")

print("\n📊 Top 10 Correlations with 'time_to_hit_hours' (when fire hit):")
time_corr = corr_matrix['time_to_hit_hours'].drop('time_to_hit_hours').abs().sort_values(ascending=False).head(10)  # type: ignore[call-overload]
print(time_corr)
print("\n💡 Higher correlation = better predictor of timing")

In [ ]:
# Visualize top correlations with targets
# Side-by-side comparison shows which features predict different aspects
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left panel: Top correlations with event (whether fire hits)
event_corr_top = corr_matrix['event'].drop(['event', 'time_to_hit_hours']).abs().sort_values(ascending=False).head(15)  # type: ignore[call-overload]
axes[0].barh(range(len(event_corr_top)), event_corr_top.values, color='#4ECDC4')
axes[0].set_yticks(range(len(event_corr_top)))
axes[0].set_yticklabels(event_corr_top.index, fontsize=9)
axes[0].set_xlabel('Absolute Correlation', fontsize=11, fontweight='bold')
axes[0].set_title('Top 15 Features Correlated with Event\n(Predicting IF fire hits)', 
                   fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')
axes[0].invert_yaxis()  # Highest at top

# Right panel: Top correlations with time_to_hit_hours (when fire hits)
time_corr_top = corr_matrix['time_to_hit_hours'].drop(['event', 'time_to_hit_hours']).abs().sort_values(ascending=False).head(15)  # type: ignore[call-overload]
axes[1].barh(range(len(time_corr_top)), time_corr_top.values, color='#F38181')
axes[1].set_yticks(range(len(time_corr_top)))
axes[1].set_yticklabels(time_corr_top.index, fontsize=9)
axes[1].set_xlabel('Absolute Correlation', fontsize=11, fontweight='bold')
axes[1].set_title('Top 15 Features Correlated with Time to Hit\n(Predicting WHEN fire hits)', 
                   fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')
axes[1].invert_yaxis()  # Highest at top

plt.tight_layout()
plt.savefig('../figures/target_correlations.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n💡 Key insights:")
print("   - Features in both panels: Predict both IF and WHEN")
print("   - Only in left panel: Predict IF but not timing")
print("   - Only in right panel: Predict timing but not occurrence")

## 6. Feature Category Analysis

In [ ]:
# Visualize feature distributions by category
# This creates a separate figure for each feature category
for category, features in feature_categories.items():
    n_features = len(features)
    n_cols = 3  # 3 columns per row
    n_rows = (n_features + n_cols - 1) // n_cols  # Calculate rows needed
    
    print(f"\n📊 Visualizing {category.upper()} category ({n_features} features)...")
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))  # type: ignore[name-defined]
    axes = axes.flatten() if n_features > 1 else [axes]
    
    for idx, feature in enumerate(features):
        # Get non-zero values for better visualization
        # Many wildfire features are zero when fire doesn't move in that direction
        data = train_df[feature]  # type: ignore[name-defined]
        non_zero_data = data[data != 0]
        zero_pct = (data == 0).sum() / len(data) * 100
        
        if len(non_zero_data) > 0:
            # Plot histogram of non-zero values
            axes[idx].hist(non_zero_data, bins=30, color='#95E1D3', edgecolor='black', alpha=0.7)
            axes[idx].set_title(f'{feature}\n({zero_pct:.1f}% zeros)', fontsize=10, fontweight='bold')
        else:
            # Feature is all zeros - just display message
            axes[idx].text(0.5, 0.5, 'All zeros', ha='center', va='center', fontsize=12)
            axes[idx].set_title(f'{feature}\n(100% zeros)', fontsize=10, fontweight='bold')
        
        axes[idx].set_xlabel('Value', fontsize=9)
        axes[idx].set_ylabel('Frequency', fontsize=9)
        axes[idx].grid(True, alpha=0.3)
    
    # Hide unused subplots (when features don't fill all grid positions)
    for idx in range(n_features, len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle(f'{category.upper()} Features', fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig(f'../figures/category_{category}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ {category.upper()} category visualized and saved")

print("\n💡 Category analysis complete!")
print("   Look for:")
print("   - High zero percentages → need binary indicators")
print("   - Similar patterns within category → potential for aggregation")
print("   - Outliers or unusual distributions → may need special handling")

## 7. Temporal Features Analysis

In [ ]:
# Temporal metadata analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Event start hour distribution
hour_counts = train_df['event_start_hour'].value_counts().sort_index()
axes[0, 0].bar(hour_counts.index, hour_counts.values, color='#4ECDC4', edgecolor='black')
axes[0, 0].set_title('Fire Start Hour Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Hour of Day (0-23)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Event start day of week
dow_counts = train_df['event_start_dayofweek'].value_counts().sort_index()
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[0, 1].bar(range(len(dow_counts)), dow_counts.values, color='#95E1D3', edgecolor='black')
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels(dow_labels)
axes[0, 1].set_title('Fire Start Day of Week', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Day of Week')
axes[0, 1].set_ylabel('Count')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Event start month
month_counts = train_df['event_start_month'].value_counts().sort_index()
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
axes[0, 2].bar(month_counts.index, month_counts.values, color='#F38181', edgecolor='black')
axes[0, 2].set_xticks(range(1, 13))
axes[0, 2].set_xticklabels([month_labels[i-1] for i in range(1, 13)], rotation=45)
axes[0, 2].set_title('Fire Start Month', fontsize=12, fontweight='bold')
axes[0, 2].set_xlabel('Month')
axes[0, 2].set_ylabel('Count')
axes[0, 2].grid(True, alpha=0.3, axis='y')

# Event rate by hour
hour_event_rate = train_df.groupby('event_start_hour')['event'].mean()
axes[1, 0].plot(hour_event_rate.index, hour_event_rate.values, marker='o', linewidth=2, 
                markersize=8, color='#4ECDC4')
axes[1, 0].set_title('Event Rate by Start Hour', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Hour of Day (0-23)')
axes[1, 0].set_ylabel('Event Rate (Proportion Hit)')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim(0, 1)

# Event rate by day of week
dow_event_rate = train_df.groupby('event_start_dayofweek')['event'].mean()
axes[1, 1].bar(range(len(dow_event_rate)), dow_event_rate.values, color='#95E1D3', edgecolor='black')
axes[1, 1].set_xticks(range(7))
axes[1, 1].set_xticklabels(dow_labels)
axes[1, 1].set_title('Event Rate by Day of Week', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Day of Week')
axes[1, 1].set_ylabel('Event Rate (Proportion Hit)')
axes[1, 1].grid(True, alpha=0.3, axis='y')
axes[1, 1].set_ylim(0, 1)

# Event rate by month
month_event_rate = train_df.groupby('event_start_month')['event'].mean()
axes[1, 2].bar(month_event_rate.index, month_event_rate.values, color='#F38181', edgecolor='black')
axes[1, 2].set_xticks(range(1, 13))
axes[1, 2].set_xticklabels([month_labels[i-1] for i in range(1, 13)], rotation=45)
axes[1, 2].set_title('Event Rate by Month', fontsize=12, fontweight='bold')
axes[1, 2].set_xlabel('Month')
axes[1, 2].set_ylabel('Event Rate (Proportion Hit)')
axes[1, 2].grid(True, alpha=0.3, axis='y')
axes[1, 2].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../figures/temporal_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTemporal patterns analyzed")

## 8. Feature Relationships with Target

In [ ]:
# Scatter plots of key features vs time_to_hit_hours, colored by event
key_features_scatter = [
    'dist_min_ci_0_5h', 'closing_speed_m_per_h', 'area_growth_rate_ha_per_h',
    'centroid_speed_m_per_h', 'alignment_abs', 'area_first_ha'
]

fig, axes = plt.subplots(3, 2, figsize=(15, 15))
axes = axes.flatten()

for idx, feature in enumerate(key_features_scatter):
    # Separate by event status
    hit = train_df[train_df['event'] == 1]  # type: ignore[name-defined]
    censored = train_df[train_df['event'] == 0]  # type: ignore[name-defined]
    
    axes[idx].scatter(censored[feature], censored['time_to_hit_hours'], 
                     alpha=0.6, s=50, c='#FF6B6B', label='Censored', edgecolors='black', linewidth=0.5)
    axes[idx].scatter(hit[feature], hit['time_to_hit_hours'], 
                     alpha=0.6, s=50, c='#4ECDC4', label='Hit', edgecolors='black', linewidth=0.5)
    
    axes[idx].set_xlabel(feature, fontsize=10, fontweight='bold')
    axes[idx].set_ylabel('Time to Hit (hours)', fontsize=10, fontweight='bold')
    axes[idx].set_title(f'{feature} vs Time to Hit', fontsize=11, fontweight='bold')
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/feature_target_relationships.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Train vs Test Distribution Comparison

In [ ]:
# Compare distributions of key features between train and test
comparison_features = [
    'area_first_ha', 'dist_min_ci_0_5h', 'closing_speed_m_per_h',
    'area_growth_rate_ha_per_h', 'centroid_speed_m_per_h', 'alignment_abs'
]

fig, axes = plt.subplots(3, 2, figsize=(15, 12))
axes = axes.flatten()

for idx, feature in enumerate(comparison_features):
    # Get non-zero values
    train_data = train_df[train_df[feature] > 0][feature]  # type: ignore[name-defined]
    test_data = test_df[test_df[feature] > 0][feature]  # type: ignore[name-defined]
    
    # Plot histograms
    axes[idx].hist(train_data, bins=30, alpha=0.6, label='Train', color='#4ECDC4', edgecolor='black')
    axes[idx].hist(test_data, bins=30, alpha=0.6, label='Test', color='#F38181', edgecolor='black')
    
    axes[idx].set_xlabel(feature, fontsize=10, fontweight='bold')
    axes[idx].set_ylabel('Frequency', fontsize=10, fontweight='bold')
    axes[idx].set_title(f'{feature}\nTrain vs Test', fontsize=11, fontweight='bold')
    axes[idx].legend(fontsize=9)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/train_test_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTrain vs Test comparison completed")

## 10. Summary Statistics Visualization

In [ ]:
# Create summary statistics table
summary_stats = []

for category, features in feature_categories.items():  # type: ignore[name-defined]
    for feature in features:
        zero_pct = (train_df[feature] == 0).sum() / len(train_df) * 100  # type: ignore[name-defined]
        non_zero_data = train_df[train_df[feature] != 0][feature]  # type: ignore[name-defined]
        
        if len(non_zero_data) > 0:
            summary_stats.append({
                'Category': category,
                'Feature': feature,
                'Zero %': f"{zero_pct:.1f}%",
                'Mean (non-zero)': f"{non_zero_data.mean():.2f}",
                'Median (non-zero)': f"{non_zero_data.median():.2f}",
                'Std (non-zero)': f"{non_zero_data.std():.2f}",
                'Corr with event': f"{train_df[[feature, 'event']].corr(method='pearson').iloc[0, 1]:.3f}"  # type: ignore[call-arg]
            })

summary_df = pd.DataFrame(summary_stats)

print("\n=== Feature Summary Statistics ===")
print(summary_df.to_string(index=False))

# Save to CSV
summary_df.to_csv('../data/feature_summary_stats.csv', index=False)
print("\nSummary statistics saved to '../data/feature_summary_stats.csv'")

## 11. Key Insights from Visualizations

### Target Variable Insights:
1. **Event Distribution**: Approximately 50% censoring rate indicates balanced survival data
2. **Time Distribution**: Hit events occur across the full 72-hour window with varying patterns
3. **Survival Curve**: Kaplan-Meier curve shows gradual decline in survival probability

### Feature Insights:
1. **High Zero Percentages**: Many features have >50% zeros, indicating sparse fire activity
2. **Distance Features**: Strong correlation with target variables, especially `dist_min_ci_0_5h`
3. **Growth Features**: Area growth metrics show high variability and potential predictive power
4. **Temporal Patterns**: Fire start time shows some patterns but not strong predictors alone

### Correlation Insights:
1. **Distance metrics** show strongest correlation with event occurrence
2. **Growth rate features** correlate with time to hit
3. **Multicollinearity** exists within feature categories (e.g., multiple growth metrics)

### Train-Test Comparison:
1. Distributions are generally similar between train and test sets
2. No major distribution shifts detected
3. Test set appears representative of training data

### Recommendations for Modeling:
1. Use survival analysis models (Cox PH, Random Survival Forests)
2. Handle high zero percentages appropriately (zero-inflated models or feature engineering)
3. Consider feature selection to address multicollinearity
4. Create interaction features between distance and growth metrics
5. Apply appropriate scaling for features with different ranges